# SAFOD solid-Earth tides: forcing, Thomas Figure 3 analogue, and response models

This notebook is the **presentation layer** for the Sherlock pipeline. PySolid, SPOTL, the transparent tide calculation, and Models A–D are executed by scripts in `scripts/tides/`; this notebook only reads products in `outputs/tides/`.

To regenerate the calculation on Sherlock:

```bash
git pull
bash RUN_ON_SHERLOCK.sh
```

Then select the kernel **SAFOD tides (.venv)** and run this notebook.

The physical hierarchy is

$$
\text{Sun/Moon}
\rightarrow
\boldsymbol{\varepsilon}(t)
\rightarrow
\boldsymbol{\sigma}(t)
\rightarrow
\text{fault traction}
\rightarrow
\Delta v/v.
$$

The tide packages constrain the first part. The larger modeling uncertainty begins with the constitutive and rock-physics steps.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "config.json").exists() and (p / "scripts/tides").exists():
            return p
    raise FileNotFoundError("Could not locate project root.")

ROOT = find_root()
OUT = ROOT / "outputs/tides"
CONFIG = json.loads((ROOT / "config.json").read_text())
print("Project root:", ROOT)
print("Results:", OUT)


## 1. Run status and provenance

A package curve is treated as a package result only when both its numerical output and provenance file exist. Missing SPOTL output is never replaced by an analytic surrogate.


In [ ]:
def load_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None

rows=[]
for name,csv_name,prov_name in [
    ("PySolid","pysolid_tides.csv","pysolid_provenance.json"),
    ("SPOTL ertid","spotl_ertid_tides.csv","spotl_provenance.json"),
    ("analytic degree-2","analytic_degree2_tides.csv","analytic_degree2_provenance.json"),
    ("Models A-D","model_results.csv","model_provenance.json"),
]:
    p=OUT/csv_name
    q=OUT/prov_name
    prov=load_json(q)
    rows.append({
        "product":name, "data":p.exists(), "provenance":q.exists(),
        "hostname":None if prov is None else prov.get("hostname"),
        "created_utc":None if prov is None else prov.get("created_utc"),
        "git_commit":None if prov is None else prov.get("git_commit"),
    })
display(pd.DataFrame(rows))


## 2. Tidal strain forcing

**PySolid** returns solid-Earth-tide displacement. The Sherlock script evaluates a small spatial stencil around SAFOD and differentiates the displacement field to recover the horizontal strain tensor.

**SPOTL `ertid`** can return extensional strain directly. We request strain along $0^\circ$, $45^\circ$, and $90^\circ$ and reconstruct

$$
\varepsilon_{NN},\qquad
\varepsilon_{EE},\qquad
\varepsilon_{NE}.
$$

Thomas et al. (2012) used SPOTL for the Parkfield calculation. For the body tide they argued that the wavelength is sufficiently long that surface strain is not significantly different from strain at 25 km depth. Their ocean-loading treatment was depth dependent; our present comparison is body tide only.


In [ ]:
def read_product(name):
    p=OUT/name
    return None if not p.exists() else pd.read_csv(p,parse_dates=["time_utc"])

pysolid=read_product("pysolid_tides.csv")
spotl=read_product("spotl_ertid_tides.csv")
analytic=read_product("analytic_degree2_tides.csv")

print("PySolid:", "not present" if pysolid is None else len(pysolid))
print("SPOTL:", "not present" if spotl is None else len(spotl))
print("analytic:", "not present" if analytic is None else len(analytic))

if pysolid is not None:
    plt.figure(figsize=(11,5))
    plt.plot(pysolid.time_utc,1e9*(pysolid.areal_strain-pysolid.areal_strain.mean()),
             linewidth=2,label="PySolid")
    if spotl is not None:
        plt.plot(spotl.time_utc,1e9*(spotl.areal_strain-spotl.areal_strain.mean()),
                 label="SPOTL ertid")
    if analytic is not None and "areal_strain" in analytic.columns:
        plt.plot(analytic.time_utc,1e9*(analytic.areal_strain-analytic.areal_strain.mean()),
                 label="transparent degree-2")
    plt.axhline(0,linewidth=.8)
    plt.ylabel("Mean-removed areal strain (nanostrain)")
    plt.xlabel("UTC")
    plt.title("SAFOD body-tide forcing, June 16–17 2026")
    plt.grid(alpha=.25); plt.legend()
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

comparison_path=OUT/"forcing_comparison.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path))


# 3. Model B: strain → elastic stress → vertical-fault traction

Thomas et al. (2012) explicitly document the architecture

$$
\text{SPOTL strain}
\rightarrow
\text{linear elastic constitutive equation}
\rightarrow
\text{stress resolved onto a vertical SAF plane}.
$$

They use a **vertical plane striking N42°W** and plot fault-normal stress (FNS) and right-lateral shear stress (RLSS) in their Figure 3. Their methods paragraph does **not** state a plane-strain closure, a plane-stress closure, or the numerical elastic constants used in Figure 3.

The previous version of our Model B imposed plane strain and then projected the resulting tensor onto the approximately $70^\circ$-dipping SAF. That step is removed: the present tide products are surface horizontal strains, so a dipping receiver plane would require a defensible full 3-D stress tensor at depth.

## 3.1 Surface strain trace

For an isotropic traction-free surface,

$$
\sigma_{DD}=0,
$$

with the full isotropic constitutive law

$$
\sigma_{ij}=2\mu\varepsilon_{ij}
+\lambda\,\mathrm{tr}(\boldsymbol{\varepsilon})\delta_{ij}.
$$

The free-surface condition implies

$$
\varepsilon_{DD}
=-\frac{\nu}{1-\nu}
\left(\varepsilon_{NN}+\varepsilon_{EE}\right),
$$

and therefore

$$
\theta\equiv\mathrm{tr}(\boldsymbol{\varepsilon})
=\frac{1-2\nu}{1-\nu}
\left(\varepsilon_{NN}+\varepsilon_{EE}\right).
$$

This use of the traction-free boundary condition is only how we recover the **surface strain trace**. It is not a plane-strain assumption at depth.

The horizontal stresses required by a vertical receiver fault are then

$$
\sigma_{NN}=2\mu\varepsilon_{NN}+\lambda\theta,
$$

$$
\sigma_{EE}=2\mu\varepsilon_{EE}+\lambda\theta,
$$

$$
\sigma_{NE}=2\mu\varepsilon_{NE}.
$$

For a vertical fault with horizontal strike and normal vectors $\mathbf{s}$ and $\mathbf{n}$,

$$
\mathrm{FNS}=\mathbf{n}^{T}\boldsymbol{\sigma}_{h}\mathbf{n},
\qquad
\mathrm{RLSS}=\mathbf{s}^{T}\boldsymbol{\sigma}_{h}\mathbf{n}.
$$

The primary SAFOD branch uses a vertical N40°W receiver plane and the site-informed elastic scenario $E=51.9$ GPa, $\nu=0.24$. The approximately $70^\circ$ SW dip is retained as geologic context but is **not used in current Model B**. A true dipping-fault calculation is deferred until a full 3-D depth-dependent strain tensor is available.

## 3.2 Thomas et al. Figure 3 analogue for June 16–17, 2026

Thomas et al. Figure 3 shows FNS (blue) and RLSS (red) resolved onto a vertical N42°W San Andreas plane. Here we compute the same two stress components for our SAFOD experiment window.

This is a **Figure-3-style analogue**, not a claim of exact numerical reproduction of their 2001 curve. Thomas et al. state that the elastic parameters used for Figure 3 were equivalent to the top layer of the Harkrider continental-shield model but do not tabulate them in paragraph 13. The benchmark therefore uses explicit $\mu=30$ GPa and $\nu=0.25$ values and labels them as a benchmark; Thomas et al. use those values elsewhere in the same paper, but that does not prove they are the exact Figure-3 constants.

Our present curve is body tide only. Thomas et al. included ocean loading but report that the body-tide contribution dominates inland and that FNS is roughly an order of magnitude larger than RLSS.

In [ ]:
if models is None:
    print("No model results yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    available = []
    for forcing in ["pysolid", "spotl"]:
        fns_col = f"{forcing}_thomas_FNS_pa"
        rlss_col = f"{forcing}_thomas_RLSS_pa"
        if fns_col in models.columns and rlss_col in models.columns:
            available.append(forcing)
            good = models[["time_utc", fns_col, rlss_col]].dropna()
            plt.figure(figsize=(11,5))
            # Match the component colors in Thomas et al. Figure 3.
            plt.plot(good.time_utc, good[fns_col]/1000.0,
                     color="navy", linewidth=1.7, label="FNS")
            plt.plot(good.time_utc, good[rlss_col]/1000.0,
                     color="red", linewidth=1.5, label="RLSS")
            plt.axhline(0, color="black", linewidth=.8)
            plt.ylabel("Tidally induced stress (kPa)")
            plt.xlabel("UTC")
            plt.title(f"Thomas et al. (2012) Figure-3-style analogue — {forcing}")
            plt.grid(alpha=.25); plt.legend()
            plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout(); plt.show()

    if not available:
        print("Thomas-analogue columns are absent. Pull latest code and rerun run_models.py.")
    else:
        rows=[]
        for forcing in available:
            fns=np.nanmax(np.abs(models[f"{forcing}_thomas_FNS_pa"]))
            rlss=np.nanmax(np.abs(models[f"{forcing}_thomas_RLSS_pa"]))
            rows.append({
                "forcing": forcing,
                "max |FNS| (kPa)": fns/1000.0,
                "max |RLSS| (kPa)": rlss/1000.0,
                "FNS/RLSS amplitude ratio": fns/rlss,
            })
        display(pd.DataFrame(rows).style.format({
            "max |FNS| (kPa)":"{:.3f}",
            "max |RLSS| (kPa)":"{:.3f}",
            "FNS/RLSS amplitude ratio":"{:.1f}",
        }))

## 3.3 Primary SAFOD Model B and the Niu transfer

The Thomas analogue is a literature benchmark. The primary SAFOD calculation uses the site-informed elastic scenario and a vertical N40°W receiver plane.

FNS is stored positive in **tension/unclamping** to preserve the Thomas convention. For the pressure-like Niu transfer we define

$$
\Delta\sigma_B=-\mathrm{FNS},
$$

so positive $\Delta\sigma_B$ means fault-normal compression, and calculate

$$
\left(\frac{\Delta v}{v}\right)_B
=S_{\mathrm{Niu}}\Delta\sigma_B,
\qquad
S_{\mathrm{Niu}}=2.4\times10^{-7}\ \mathrm{Pa}^{-1}.
$$

That last multiplication is **our transfer assumption**. Niu et al. measured a local SAFOD barometric-pressure sensitivity; they did not publish a universal tidal FNS sensitivity. The code therefore also saves the alternative RLSS $\rightarrow$ Niu result as a sensitivity branch.

In [ ]:
if models is not None:
    for forcing in ["pysolid", "spotl"]:
        fns = f"{forcing}_FNS_tension_pa"
        rlss = f"{forcing}_RLSS_pa"
        if fns in models.columns and rlss in models.columns:
            good=models[["time_utc", fns, rlss]].dropna()
            plt.figure(figsize=(11,5))
            plt.plot(good.time_utc, good[fns]/1000.0, label="FNS (tension +)")
            plt.plot(good.time_utc, good[rlss]/1000.0, label="RLSS")
            plt.axhline(0, linewidth=.8)
            plt.ylabel("Stress perturbation (kPa)")
            plt.xlabel("UTC")
            plt.title(f"Revised Model B — vertical SAFOD receiver plane ({forcing})")
            plt.grid(alpha=.25); plt.legend()
            plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
            plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

## 3.4 Mechanical scope of Model B

The revised Model B is intentionally narrower than the old calculation:

$$
\boxed{
\text{surface body-tide strain}
\rightarrow
\text{surface-derived horizontal elastic stress}
\rightarrow
\text{vertical SAF traction}
}
$$

It is sufficient for the Thomas-Figure-3-style FNS/RLSS diagnostic and avoids pretending that we know the full stress tensor on a $70^\circ$-dipping plane at approximately 1 km depth.

A rigorous dipping-fault calculation should instead start from a **full 3-D depth-dependent strain tensor** and then apply isotropic or anisotropic elasticity at depth. That is the next mechanical refinement, not another arbitrary plane-stress/plane-strain closure.

## 3.3 Stress → $\Delta v/v$ is a separate assumption

The elastic calculation ends at the tidal stress history. Turning that stress into a seismic-velocity perturbation is **not supplied by Thomas et al.**

The present empirical branch applies the SAFOD coefficient from Niu et al. (2008),

$$
S_{\mathrm{Niu}}=2.4\times10^{-7}\ \mathrm{Pa}^{-1},
$$

through

$$
\left(\frac{\Delta v}{v}\right)_B
=
S_{\mathrm{Niu}}\,\Delta\sigma_*.
$$

Using Niu's barometric stress sensitivity for a tidal stress component is an **assumed cross-loading transfer**. It should be kept conceptually separate from the much better constrained tidal strain and elastic-stress calculation.


# 4. Models A–D and AWD detectability

Model A is the Niu 240-Pa amplitude shortcut. Model B is the explicit elastic-stress calculation above followed by the Niu transfer. Model C applies direct strain sensitivities from other sites as context. Model D uses a stress-dependent crack-compliance model.

No branch constitutes a tidal detection.


In [ ]:
if models is not None:
    summary_rows=[]
    for forcing in ["pysolid","spotl"]:
        mapping={
            "A":f"{forcing}_model_A_dv_over_v",
            "B":f"{forcing}_model_B_dv_over_v",
            "C (Takano)":f"{forcing}_model_C_takano_dv_over_v",
            "D (Vs)":f"{forcing}_model_D_dVs_over_Vs",
            "D (Vp)":f"{forcing}_model_D_dVp_over_Vp",
        }
        if all(col in models.columns for col in mapping.values()):
            for name,col in mapping.items():
                amp=np.nanmax(np.abs(models[col]))
                summary_rows.append({
                    "forcing":forcing,
                    "model":name,
                    "max_abs_dv/v":amp,
                    "max_abs_percent":100*amp,
                    "Deep reliable / model":CONFIG["awd_benchmarks"]["deep_outbound_reliable"]/amp,
                })
    summary=pd.DataFrame(summary_rows)
    display(summary.style.format({
        "max_abs_dv/v":"{:.3e}",
        "max_abs_percent":"{:.5f}",
        "Deep reliable / model":"{:.1f}x",
    }))


## 5. Interpretation and limitations

The modeling hierarchy is deliberately explicit:

$$
\boxed{\text{tide packages}}
\rightarrow
\boxed{\boldsymbol{\varepsilon}(t)}
\rightarrow
\boxed{\text{elastic closure}}
\rightarrow
\boxed{\text{fault stress}}
\rightarrow
\boxed{\text{stress-to-velocity model}}.
$$

The PySolid/SPOTL comparison addresses forcing uncertainty. The Thomas-style Figure 3 analogue checks whether the stress construction behaves sensibly. The SAFOD geometry then supplies the site-specific stress scenario. The final stress-to-$\Delta v/v$ step remains the least constrained part.

Important limitations:

- The current Model B does **not** impose plane strain. It uses a free-surface condition to recover the surface strain trace, applies isotropic Hooke law to the horizontal stresses, and resolves traction on a vertical SAF plane.
- This surface-derived calculation is a stress benchmark, not the exact 3-D tidal stress tensor along the approximately 1 km-deep DAS interval. A dipping-fault calculation requires a full 3-D depth-dependent strain tensor.
- Niu's coefficient is an empirical barometric sensitivity, not a universal tidal coefficient.
- Model D predicts formation-scale $V_P/V_S$, not the AWD guided apparent velocity.
- No tidal response is claimed to have been detected in the AWD experiment.

## References

- Thomas et al. (2012), *JGR Solid Earth*, DOI `10.1029/2011JB009036`.
- van der Elst et al. (2016), *PNAS*, DOI `10.1073/pnas.1524316113`.
- Agnew (2012), SPOTL.
- Niu et al. (2008), *Nature*, DOI `10.1038/nature07111`.
